# Notebook 02k - PPO-CF on `MiniGrid-KeyCorridorS3R1-v0`

Compare PPO and PPO-CF from `config/envs/keycorridors3r1_cf.yaml`, changing
only `ppo.pg_mode` between arms. The base PPO hyperparameters are the mapped RL
Baselines3 Zoo KeyCorridor settings; the CF arm adds branch rollouts from exact
MiniGrid simulator states.


---
## Knobs


In [ ]:
# ----------------------------------------------------------------------------
# EDIT ME
# ----------------------------------------------------------------------------
ENV_CONFIG    = "keycorridors3r1_cf"
SEEDS         = None          # None -> use the config's seeds
FORCE_RETRAIN = True

RUN_ORACLE_CHECKS = True
RUN_CONTROL       = True
RUN_CF            = True
RUN_SHUFFLED      = False

ORACLE_CHECK_RUN  = "keycorridors3r1"
ORACLE_CHECK_CKPT = 0.75

OVERRIDES = {
    # "ppo.total_timesteps": 100_000,
    # "run.seeds": (0,),
    # "ppo.cf_horizon": 32,
    # "ppo.cf_subsample": 0.10,
    # "ppo.alpha_cf": 0.25,
    # Fixed-layout/action-floor fallback:
    # "env.layout_seeds": (0,),
    # "ppo.prob_floor_start": 0.15,
    # "ppo.prob_floor_end": 0.02,
    # "ppo.ent_coef": 0.03,
}
# ----------------------------------------------------------------------------


## 0. Setup


In [ ]:
%load_ext autoreload
%autoreload 2

import sys, pathlib, time

ROOT = pathlib.Path.cwd()
if not (ROOT / "config").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from config import make_config, seed_dir, RUNS_DIR, FIGURES_DIR
from dataio import load_trajectories, load_checkpoint, list_checkpoints
from utils.logging import read_scalars
from utils.plotting import savefig
from scripts.train import run_seeds

base = make_config(ENV_CONFIG, **OVERRIDES)
if SEEDS is not None:
    base = make_config(ENV_CONFIG, **{**OVERRIDES, "run.seeds": tuple(SEEDS)})
SEEDS = tuple(base.run.seeds)
STEM = base.run.run_name

ARMS = {
    k: make_config(ENV_CONFIG, **{**OVERRIDES, "run.seeds": SEEDS,
                                  "ppo.pg_mode": m, "run.run_name": f"{STEM}_{k}"})
    for k, m in [("gae", "gae"), ("cf", "cf_all_action"), ("shuf", "cf_shuffled")]
}
FIG = FIGURES_DIR / f"nb02_{STEM}"
FIG.mkdir(parents=True, exist_ok=True)
ACTION_NAMES = ["left", "right", "forward", "pickup", "drop", "toggle", "done"]

print(ARMS["cf"].summary())
print()
print(f"arms: {[c.run.run_name for c in ARMS.values()]}   seeds: {list(SEEDS)}")


---
# 1. Test The Oracle

The config uses `cf_restore: fast` for speed, while `cf_validate: true` checks
recorded transitions before training trusts the restore path.


In [ ]:
from envs.env_pool import set_sim_state
from oracle.online import OnlineOracle, check_replay, check_centering, landscape_summary

if RUN_ORACLE_CHECKS:
    cfg = ARMS["cf"]
    K = len(ACTION_NAMES)
    sd = seed_dir(ORACLE_CHECK_RUN, SEEDS[0]) if ORACLE_CHECK_RUN else None
    traj_path = sd / "trajectories.npz" if sd else None
    if sd and traj_path.exists():
        traj = load_trajectories(traj_path)
        ckpts = list_checkpoints(sd / "checkpoints")
        if not ckpts:
            raise FileNotFoundError(f"no checkpoints found under {sd / 'checkpoints'}")
        loaded = [(p, load_checkpoint(p)) for p in ckpts]
        p_ck, ck = min(loaded, key=lambda pc: abs(pc[1].fraction - ORACLE_CHECK_CKPT))
        value_fn, probs_fn = ck.values, ck.probs
        src = f"{ORACLE_CHECK_RUN} @ {ck.fraction:.0%} ({p_ck.name})"
        N = min(800, len(traj))
        sims  = traj.sim_state[:N]
        acts  = traj.action[:N]
        rews  = traj.reward[:N]
        nsims = traj.next_sim_state[:N]
    else:
        from agents.ppo import PPOTrainer
        print(f"no completed trajectory at {traj_path}; using one live rollout for smoke-only oracle checks")
        _t = PPOTrainer(cfg, seed=SEEDS[0], progress=False)
        _t.pool.reset(); _t.collect_rollout()
        T, NE = cfg.ppo.n_steps, cfg.env.n_envs
        f = lambda a: a[:T].reshape((T * NE,) + a.shape[2:])
        sims, acts, rews, nsims = f(_t.buffer.sim_state), f(_t.buffer.actions), f(_t.buffer.rewards), f(_t.buffer.next_sim_state)
        value_fn, probs_fn = _t._values_np, _t._probs_np
        src = "fresh rollout"

    oracle = OnlineOracle(cfg.env.env_id, K, cfg.ppo.gamma,
                          {"fully_observable": cfg.env.fully_observable},
                          cfg.env.max_episode_steps, restore=cfg.ppo.cf_restore)
    print(f"oracle check source: {src}; {len(sims)} states; K={K}")


## 1.1 Restore, Centering, And Landscape Shape


In [ ]:
if RUN_ORACLE_CHECKS:
    rep = check_replay(oracle, sims[:400], acts[:400], rews[:400], nsims[:400])
    print(f"restore replay exact over {rep['n']} transitions")
    print(f"  max reward error {rep['max_reward_error']:.3e}")
    print(f"  max state error  {rep['max_state_error']:.3e}")

    obs0 = np.stack([set_sim_state(oracle.env, s, elapsed_steps=0) for s in sims[:300]])
    pi = probs_fn(obs0)
    a_cf, q_cf, diag = oracle.a_g(
        sims[:300], pi[:300], probs_fn, value_fn,
        horizon=cfg.ppo.cf_horizon, n_rollouts=cfg.ppo.cf_rollouts,
        bootstrap_tail=cfg.ppo.cf_bootstrap_tail, chunk=cfg.ppo.cf_branch_envs)
    c = check_centering(a_cf, pi[:300])
    s_ = landscape_summary(a_cf, q_cf, pi[:300], spread_threshold=cfg.oracle.spread_threshold)

    print()
    print("Q_g diagnostics")
    print(f"  branches that saw reward         {diag['reward_coverage']:.3f}")
    print(f"  branches terminating within H    {diag['terminated_within_horizon']:.3f}")
    print(f"  states with reward in some branch {diag['frac_states_any_reward']:.3f}")
    print()
    print("Landscape")
    print(f"  centering max |sum pi*A_CF|      {c['max_abs']:.3e}")
    print(f"  mean |A_CF|                      {s_['mean_abs_a_cf']:.4f}")
    print(f"  frac states with spread > {cfg.oracle.spread_threshold}    {s_['frac_states_with_spread']:.3f}")
    print(f"  all finite                       {s_['finite']}")
    print("  best action by A_CF:")
    for name, count in zip(ACTION_NAMES, s_["best_action_counts"]):
        print(f"    {name:8s} {count:5d}")


## 1.2 Oracle Verdict


In [ ]:
if RUN_ORACLE_CHECKS:
    assert rep["max_state_error"] == 0.0
    assert rep["max_reward_error"] == 0.0
    assert c["max_abs"] < 1e-4
    assert s_["finite"]
    print("oracle checks passed")


---
# 2. Train Arms


In [ ]:
def train_arm(key):
    c = ARMS[key]
    done = all((seed_dir(c.run.run_name, s) / "scalars.csv").exists() for s in c.run.seeds)
    if done and not FORCE_RETRAIN:
        print(f"found existing run at {RUNS_DIR / c.run.run_name} -- skipping")
        return None
    t0 = time.time()
    out = run_seeds(c)
    print()
    print(f"{key}: {(time.time() - t0) / 60:.1f} min total")
    return out

if RUN_CONTROL:
    train_arm("gae")
if RUN_CF:
    train_arm("cf")
if RUN_SHUFFLED:
    train_arm("shuf")


---
# 3. Compare Arms


In [ ]:
scal = {k: {s: read_scalars(seed_dir(c.run.run_name, s) / "scalars.csv")
            for s in c.run.seeds if (seed_dir(c.run.run_name, s) / "scalars.csv").exists()}
        for k, c in ARMS.items()}
scal = {k: v for k, v in scal.items() if v}

THRESH = 0.8
def frames_to(d, col="success_rate_100", thresh=THRESH):
    hit = d.loc[d[col] >= thresh, "global_step"]
    return int(hit.iloc[0]) if len(hit) else None

rows = []
for arm, per_seed in scal.items():
    for s, d in per_seed.items():
        rows.append({
            "arm": arm,
            "seed": s,
            f"frames_to_{THRESH:g}": frames_to(d),
            "final_return": round(float(d["mean_return_100"].iloc[-1]), 3),
            "final_success": round(float(d["success_rate_100"].iloc[-1]), 3),
            "best_success": round(float(d["success_rate_100"].max()), 3),
            "final_sg1": round(float(d["subgoal1_rate_100"].iloc[-1]), 3) if "subgoal1_rate_100" in d else None,
            "final_sg2": round(float(d["subgoal2_rate_100"].iloc[-1]), 3) if "subgoal2_rate_100" in d else None,
            "entropy_last": round(float(d["entropy"].iloc[-1]), 3),
            "ev_median": round(float(d["explained_variance"].median()), 3),
            "kl_median": round(float(d["approx_kl"].median()), 4),
            "clipfrac_median": round(float(d["clipfrac"].median()), 3),
            "cf_coverage": (round(float(d["cf_reward_coverage"].mean()), 3)
                            if "cf_reward_coverage" in d else None),
            "cf_corr_gae": (round(float(d["cf_corr_gae"].mean()), 3)
                            if "cf_corr_gae" in d else None),
        })
cmp_df = pd.DataFrame(rows).sort_values(["arm", "seed"])
display(cmp_df)

print(f"median frames to success >= {THRESH:g}")
for arm, g in cmp_df.groupby("arm") if not cmp_df.empty else []:
    v = g[f"frames_to_{THRESH:g}"].dropna()
    print(f"  {arm:4s} {('%.0f' % v.median()) if len(v) else 'never':>10s}   ({len(v)}/{len(g)} seeds reached it)")


## 3.1 Curves And CF Health


In [ ]:
if scal:
    fig, axes = plt.subplots(1, 5, figsize=(22, 3.8))
    colors = {"gae": "#3b6ea5", "cf": "#c4622d", "shuf": "#7a7a7a"}
    panels = [("success_rate_100", "Success rate"),
              ("mean_return_100", "Return"),
              ("subgoal1_rate_100", "Key picked"),
              ("subgoal2_rate_100", "Locked door open"),
              ("cf_reward_coverage", "Q_g reward coverage")]
    for ax, (col, title) in zip(axes, panels):
        for arm, per_seed in scal.items():
            for i, (s, d) in enumerate(per_seed.items()):
                if col not in d.columns or not np.isfinite(d[col]).any():
                    continue
                ax.plot(d["global_step"], d[col], lw=1.4, alpha=0.85, color=colors[arm],
                        label=arm if i == 0 else None)
        ax.set_title(title, fontsize=10)
        ax.set_xlabel("environment steps")
        ax.spines[["top", "right"]].set_visible(False)
        ax.grid(axis="y", alpha=0.25)
    axes[0].axhline(THRESH, color="#888", ls=":", lw=1)
    for ax in axes[:4]:
        ax.set_ylim(-0.05, 1.05)
    axes[0].legend(frameon=False, fontsize=9)
    fig.tight_layout()
    savefig(fig, FIG / "arms.png")
    plt.show()

    for arm in ("cf", "shuf"):
        if arm not in scal:
            continue
        seed0 = sorted(scal[arm])[0]
        d = scal[arm][seed0]
        print()
        print(f"{ARMS[arm].run.run_name}  (seed {seed0})")
        for col in ["cf_centering", "cf_reward_coverage", "cf_corr_gae", "cf_mean_abs", "cf_n_cf_states"]:
            if col not in d.columns or not np.isfinite(d[col]).any():
                print(f"  {col:20s} not logged")
                continue
            print(f"  {col:20s} mean {d[col].mean():+.4g}  last {d[col].iloc[-1]:+.4g}")


## 3.2 Action Collapse Check


In [ ]:
def load_action_summary(run_name, seeds):
    rows = []
    for s in seeds:
        path = seed_dir(run_name, s) / "trajectories.npz"
        if not path.exists():
            continue
        traj = load_trajectories(path)
        acts = traj.action.astype(int)
        probs = traj.probs.astype(float)
        for a, name in enumerate(ACTION_NAMES):
            rows.append({
                "run": run_name,
                "seed": s,
                "action": name,
                "sample_frac": float((acts == a).mean()),
                "mean_pi": float(probs[:, a].mean()),
                "p05_pi": float(np.quantile(probs[:, a], 0.05)),
            })
    return rows

action_rows = []
for arm, c in ARMS.items():
    if arm == "shuf" and not RUN_SHUFFLED:
        continue
    action_rows.extend(load_action_summary(c.run.run_name, c.run.seeds))
action_df = pd.DataFrame(action_rows)
display(action_df)

if not action_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
    for ax, val, title in [(axes[0], "mean_pi", "Mean policy probability"),
                           (axes[1], "sample_frac", "Sampled action fraction")]:
        piv = action_df.pivot_table(index="action", columns="run", values=val, aggfunc="mean").loc[ACTION_NAMES]
        piv.plot(kind="bar", ax=ax)
        ax.set_title(title)
        ax.set_ylabel(val)
        ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    savefig(fig, FIG / "action_compare.png")
    plt.show()


## 4. Reading The Result

A useful PPO-CF improvement should show higher success or faster time-to-success
without KL/clipfrac spikes, and without `pickup` or `toggle` collapsing to near
zero probability. If Q_g reward coverage is near zero at H=64, increase horizon
before interpreting the CF arm.
